# Finding metacells with SEACell

- [notebook from where this one here is copied and adapted from](https://github.com/dpeerlab/SEACells/blob/main/notebooks/SEACell_computation.ipynb)^
- [paper of tool](https://doi.org/10.1038/s41587-023-01716-9)
- [paper reviewing and explaining the use of metacells + recommendations](https://doi.org/10.1038/s44320-024-00045-6)

![](https://cdn.ncbi.nlm.nih.gov/pmc/blobs/1136/11220014/ee44a48fe168/44320_2024_45_Fig8_HTML.jpg)
> Metacells increase profile coverage and save computational resources, while preserving biologically relevant heterogeneity in single-cell genomics data.

>This notebook is a tutorial for computing SEACell metacells, visualizing results and computing evaluation metrics

## Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

import SEACells

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# Some plotting aesthetics
%matplotlib inline

sns.set_style('ticks')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100

## Load Data

>We recommend the use of scanpy Anndata objects as the preferred mode of loading and filtering data.

In [ ]:
input_file = "/data/cephfs-1/home/users/cemo10_c/work/scRNA/scRNA_preprocessing_pipeline/results/per_sample/CE_SC_5FU_Conti_5/adata_ready_for_merge_counts__theislab_tutorial.h5ad"
output_file = "/data/cephfs-1/home/users/cemo10_c/work/scRNA/scRNA_preprocessing_pipeline/results/per_sample/CE_SC_5FU_Conti_5/adata_ready_for_merge_SEACell_metacells.h5ad"
count_layer = 'counts'
normalization = 'log1p_norm'

In [ ]:
# Load the data using 
ad = sc.read(input_file)
ad.X = ad.layers[count_layer]
ad

In [ ]:
# Plot cell-types for reference
sc.pl.scatter(ad, basis='umap', color='leiden_res0_5', frameon=False)

## Pre-processing

>The following section describes basic pre-processing steps for scRNA-seq.

In [ ]:
# Copy the counts to ".raw" attribute of the anndata since it is necessary for downstream analysis
# This step should be performed after filtering 
raw_ad = sc.AnnData(ad.X)
raw_ad.obs_names, raw_ad.var_names = ad.obs_names, ad.var_names
ad.raw = raw_ad

In [ ]:
layer_name = normalization + "_of_" + count_layer
ad.X = ad.layers[layer_name]

# Normalize cells, log transform and compute highly variable genes
# sc.pp.normalize_per_cell(ad)
# sc.pp.log1p(ad)
sc.pp.highly_variable_genes(ad, n_top_genes=1500)

In [ ]:
# Compute principal components - 
# Here we use 50 components. This number may also be selected by examining variance explaint
sc.tl.pca(ad, n_comps=50, use_highly_variable=True)

In [ ]:
ad

# Running SEACells

>As a rule of thumb, we recommended choosing one metacell for every 75 single-cells.
>
><b>Note 1: </b> Running SEACells modifies the input Anndata object and adds the SEACell metacell assignments to the `obs` dataframe in the anndata object.
>
><b>Note 2: </b> This analysis takes approxmiately 5 minutes

In [ ]:
## User defined parameters

## Core parameters 
n_SEACells = round(ad.shape[0] / 30)
build_kernel_on = 'X_pca' # key in ad.obsm to use for computing metacells
                          # This would be replaced by 'X_svd' for ATAC data

## Additional parameters
n_waypoint_eigs = 10 # Number of eigenvalues to consider when initializing metacells

In [ ]:
model = SEACells.core.SEACells(ad, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

In [ ]:
model.construct_kernel_matrix()
M = model.kernel_matrix

In [ ]:
sns.clustermap(M.toarray()[:500,:500])

In [ ]:
# Initialize archetypes
model.initialize_archetypes()

In [ ]:
# Plot the initilization to ensure they are spread across phenotypic space
SEACells.plot.plot_initialization(ad, model)

In [ ]:
model.fit(min_iter=10, max_iter=150)

In [ ]:
# # You can force the model to run additional iterations step-wise using the .step() function
# print(f'Ran for {len(model.RSS_iters)} iterations')
# for _ in range(5):
#     model.step()
# print(f'Ran for {len(model.RSS_iters)} iterations')

## Accessing results

### Model Convergence

In [ ]:
# Check for convergence 
model.plot_convergence()

### SEACell Hard Assignments

>These can be accessed as folows:
>- in the modified anndata object in `.obs['SEAell']` 
>- from the model using `.get_hard_assignments()` 


In [ ]:
ad.obs[['SEACell']].head()

In [ ]:
model.get_hard_assignments().head()

### SEACell Soft Assignments

>Archetypal analysis returns soft assignments of cells to SEACells. The full assignment matrix can be accessed as the parameter ```model.A_```. However, the majority of single-cells are assigned to no more than 4 archetypes with non-trivial weight, so we return the top 5 metacell assignments as well as the corresponding assignment weights in the function ```model.get_soft_assignments()```

In [ ]:
plt.figure(figsize=(3,2))
sns.distplot((model.A_.T > 0.1).sum(axis=1), kde=False)
plt.title(f'Non-trivial (> 0.1) assignments per cell')
plt.xlabel('# Non-trivial SEACell Assignments')
plt.ylabel('# Cells')
plt.show()

plt.figure(figsize=(3,2))
b = np.partition(model.A_.T, -5)    
sns.heatmap(np.sort(b[:,-5:])[:, ::-1], cmap='viridis', vmin=0)
plt.title('Strength of top 5 strongest assignments')
plt.xlabel('$n^{th}$ strongest assignment')
plt.show()


In [ ]:
labels,weights = model.get_soft_assignments()

In [ ]:
labels.head()

## Summarizing data

>- `core.summarize_by_SEACell()`
>
>Datasets can be summarized by SEACell by aggregating cells within each SEACell, summing over all raw data for all cells belonging to a SEACell. The output of this function is an anndata object of shape n_metacells x original_data_dimension. Data is unnormalized and raw aggregated counts are stored in `X`. Attributes associated with variables (.var) are copied over, but relevant per SEACell attributes must be manually copied, since certain attributes may need to be summed, or averaged etc, depending on the attribute.
>
>By default, `ad.raw` is used for summarization. Other layers present in the anndata can be specified using the parameter `summarize_layer` parameter

In [ ]:
SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')
SEACell_ad

In [ ]:
# add to SEACell_ad the number of cells per metacell/SEACell

# Count how many cells are assigned to each metacell
metacell_counts = ad.obs['SEACell'].value_counts().to_dict()

# Ensure metacell order matches between SEACell_ad and the counts
SEACell_ad.obs['n_cells'] = SEACell_ad.obs_names.map(metacell_counts).fillna(0).astype(int)

# Verify the addition
print(SEACell_ad.obs['n_cells'])

>Normalization of metacell data can be performed using the `sc.pp.normalize_total` and `sc.pp.log1p` functions

In [ ]:
SEACell_ad.layers['counts'] = SEACell_ad.X.copy()
sc.pp.normalize_total(SEACell_ad, target_sum=1e4)
sc.pp.log1p(SEACell_ad)
SEACell_ad.layers['log1p_norm' + "_of_" + count_layer] = SEACell_ad.X.copy()

In [ ]:
SEACell_ad.write_h5ad(output_file)

# are the counts still integers as required by DeSeq2 later on? (They should be unless soupx_counts were used)
print(SEACell_ad.X[0:5, 0:5].todense())

>Alternatively, we can take into account soft assignments of cells to SEACells by weighting cells by the strength of the assignment. A minimum assignment weight is used to zero out trivial assignments

In [ ]:
SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(ad, model.A_, celltype_label='leiden_res0_5',summarize_layer='raw', minimum_weight=0.05)
SEACell_soft_ad

In [ ]:
SEACell_soft_ad.obs.head()

## Evaluating Results

>We provide several methods for evaluating SEACell assignments:

### Visualizing Results

>Metacells also implements methods for visualizing the results of the Metacells algorithm 
    <ul> 
        <li>```.plot_2D()``` provides an interface for viewing metacell assignments on any 2-dimensional embedding in ad.obsm. Plots can also be coloured by metacell assignment.
        <li>```.plot_SEACell_sizes()``` can be used to view the distribution of number of cells assigned to each metacell
    </ul>
    
            

In [ ]:
SEACells.plot.plot_2D(ad, key='X_umap', colour_metacells=False)

In [ ]:
SEACells.plot.plot_2D(ad, key='X_umap', colour_metacells=True)

In [ ]:
SEACells.plot.plot_SEACell_sizes(ad, bins=5)

### Quantifying Results

>SEACells also implements methods for visualizing the results of the SEACells algorithm 
    <ul> 
        <li>```.compute_celltype_purity(ad, col_name)``` computes the purity of different celltype labels within a SEACell metacell. Typically, col_name='celltype' or similar. Returns a pd.DataFrame of length n_SEACells.
        <li>```.compactness(ad, low_dim_embedding)``` computes the per-SEAcell variance in diffusion components. ```low_dim_embedding``` is a string specifying the low dimensional embedding with which diffusion components are calculated, typically 'X_pca' for RNA or 'X_svd' for ATAC. Lower values of compactness suggest more compact/lower variance metacells.
        <li>```separation(ad, low_dim_embedding,nth_nbr=1,cluster=None)``` computes the diffusion distance between a SEACell and its ```nth_nbr```. As before, ```low_dim_embedding``` is a string specifying the low dimensional embedding with which diffusion components are calculated, typically 'X_pca' for RNA or 'X_svd' for ATAC. If ```cluster``` is provided as a string, e.g. 'celltype', nearest neighbors are restricted to have the same celltype value.  Higher values of separation suggest better distinction between metacells.
    </ul>
    


In [ ]:
SEACell_purity = SEACells.evaluate.compute_celltype_purity(ad, 'leiden_res0_5')

plt.figure(figsize=(4,4))
sns.boxplot(data=SEACell_purity, y='leiden_res0_5')
plt.title('Celltype Purity')
sns.despine()
plt.show()
plt.close()

SEACell_purity.head()

In [ ]:
compactness = SEACells.evaluate.compactness(ad, 'X_pca')

plt.figure(figsize=(4,4))
sns.boxplot(data=compactness, y='compactness')
plt.title('Compactness')
sns.despine()
plt.show()
plt.close()

compactness.head()

In [ ]:
separation = SEACells.evaluate.separation(ad, 'X_pca',nth_nbr=1)

plt.figure(figsize=(4,4))
sns.boxplot(data=separation, y='separation')
plt.title('Separation')
sns.despine()
plt.show()
plt.close()

separation.head()

In [ ]:
import umap as py
py.__version__

In [ ]:
# are the counts still integers as required by DeSeq2 later on? (They should be unless soupx_counts were used)
print(ad.X[0:5, 0:5].todense())
print(ad.layers['counts'][0:5, 0:5].todense())
ad.raw[0:5, 0:5]